In [1]:
import os
import sys

# 1. Clone the repository (if not already there)
repo_name = "FINRL"
repo_url = "https://github.com/nidarshans/FINRL.git" # Use HTTPS for Colab

if not os.path.exists(repo_name):
    print(f"Cloning {repo_name}...")
    !git clone {repo_url}
else:
    print(f"{repo_name} already exists. Pulling latest changes...")
    %cd {repo_name}
    !git pull
    %cd ..

%cd /content/FINRL/


# 3. Verify imports
try:
    import pandas as pd
    from lib.regime_detection.src.constants import *
    print("✅ Imports successful!")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("Current directory contents:", os.listdir())



Cloning FINRL...
Cloning into 'FINRL'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 24 (delta 0), reused 24 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 12.40 KiB | 6.20 MiB/s, done.
/content/FINRL
✅ Imports successful!


In [2]:
# ==============================================================================
# Sector Rotation Regime Detection Model — Modular Sub-package Implementation
# ==============================================================================

import os
import sys
import warnings
warnings.filterwarnings('ignore')


%pip install yfinance hmmlearn pandas numpy matplotlib scipy pandas-ta scikit-learn bt plotly kaleido

try:
    import pandas as pd
    from lib.regime_detection.src.constants import *
    from lib.regime_detection.src.data.loader import download_all, _slice_data
    from lib.regime_detection.src.execution.backtest import train_all_sectors, decode_test_sectors, build_weight_matrix, run_bt_backtest
    from lib.regime_detection.src.execution.walk_forward import run_walk_forward
    from lib.regime_detection.src.utils.plotting import plot_results, print_stats
except ImportError as e:
    print(f"Import failed: {e}")
    print("Current sys.path:", sys.path)
    raise e

def main():
    all_tickers = SECTORS + [BENCHMARK]

    if WALK_FORWARD:
        all_data = download_all(all_tickers, WF_FULL_START, WF_FULL_END)
        (wf_weights, wf_decoded, wf_windows, all_close, bench_series) = run_walk_forward(all_data)
        result, close_prices, weights_aligned = run_bt_backtest(
            wf_weights, all_close, bench_series,
            label=f"HMM Walk-Forward ({WF_MODE})"
        )
        print_stats(result)
        mode_label = f"Walk-Forward {WF_MODE.capitalize()} | train={WF_TRAIN_DAYS}d oos={WF_OOS_DAYS}d"
        plot_results(result, weights_aligned, wf_decoded, close_prices, "sector_rotation_wf.png", wf_windows=wf_windows, mode_label=mode_label)

    else:
        all_data = download_all(all_tickers, TRAIN_START, TEST_END)
        train_data = _slice_data(all_data, TRAIN_START, TRAIN_END)
        test_data = _slice_data(all_data, TEST_START, TEST_END)
        benchmark_test = test_data.pop(BENCHMARK, None)
        if benchmark_test is None: raise RuntimeError("Benchmark SPY missing.")
        
        trained = train_all_sectors(train_data)
        decoded_test = decode_test_sectors(test_data, trained)
        weights = build_weight_matrix(decoded_test)
        close_prices = pd.DataFrame({t: df["Close"] for t, df in test_data.items() if t in weights.columns}).ffill().dropna(how="all")
        bench_series = benchmark_test["Close"].reindex(close_prices.index).ffill()
        
        result, close_prices, weights_aligned = run_bt_backtest(weights, close_prices, bench_series, label="HMM Sector Rotation")
        print_stats(result)
        mode_label = f"Train {TRAIN_START}→{TRAIN_END} | Test {TEST_START}→{TEST_END}"
        plot_results(result, weights_aligned, decoded_test, close_prices, "sector_rotation_results.png", mode_label=mode_label)

    print("\nDone!")
    return result

if __name__ == "__main__":
    result = main()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 63.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 78.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 59.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 17.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling nu

ModuleNotFoundError: No module named 'bt'